In [1]:
import numpy as np
from dask.distributed import Client
import project_path

from src.synthetic_data.generate_lr_data import generate_low_rank_data
from src.multilinear_ops.merge_tucker import merge_tucker
from src.multilinear_ops.mode_product import mode_product
from src.models.tucker_decomp.hosvd import HoSVD
from src.models.tucker_decomp.hooi import HoOI

/mnt/ufs18/home-207/indibimu/repos/ML_GSP


In [3]:
# client = Client(n_workers=4)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 20.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:37422,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 16
Started: Just now,Total memory: 20.00 GiB
Comm: tcp://127.0.0.1:38399,Total threads: 4
Dashboard: http://127.0.0.1:46800/status,Memory: 5.00 GiB
Nanny: tcp://127.0.0.1:43991,


In [6]:
%%timeit -n 1 -r 1
futures = []
X = generate_low_rank_data((400,400,400), (5,6,7))
h = HoSVD(X)
C, Us = h()

1min ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [5]:
%%timeit -n 1 -r 1
futures = []
X = generate_low_rank_data((400,400,400), (5,6,7))
h = HoSVD(X, client=client)
C, Us = h()

/mnt/home/indibimu/miniconda3/envs/ML_GSP/lib/python3.12/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 488.28 MiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


34.3 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [7]:
%%timeit -n 1 -r 1
futures = []
X = generate_low_rank_data((500,500,500), (5,6,7), seed=10)
# h = HoSVD(X, client=client)
h = HoSVD(X, n_ranks=(5,6,7))
C, Us = h()
print(np.isclose(X, merge_tucker(C, Us, np.arange(3))).all())

True
7.01 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [8]:
%%timeit -n 1 -r 1
futures = []
X = generate_low_rank_data((500,500,500), (5,6,7))
h = HoSVD(X, n_ranks=(5,6,7), client=client)
# h = HoSVD(X)
C, Us = h()

/mnt/home/indibimu/miniconda3/envs/ML_GSP/lib/python3.12/site-packages/distributed/client.py:3163: UserWarning: Sending large graph of size 0.93 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


12.9 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [6]:
X = generate_low_rank_data((200,200,200), (5,6,7), seed=10)
hooi = HoOI(X, (7,7,7), max_it=40)
C, Us = hooi()
print(np.isclose(X, merge_tucker(C, Us, np.arange(3))).all())

Iteration 0: ||X|| - ||C|| = -8.526512829121202e-14
True


In [ ]:
client.shutdown()